# STL_to_inp_ftetwild

**What this notebook showcases in `gyroid_utils`:** the tetrahedral-meshing step of the simulation workflow. It loads a gyroid STL (`io_ops.py`), calls the external fTetWild tool to convert the surface mesh into a volumetric tetrahedral mesh, and converts that mesh into an ABAQUS input (`.inp`) file, preparing the structure for finite element simulation.


In [ ]:
import numpy as np
from stl import mesh as stl_mesh
import plotly.graph_objects as go  # for visualization
import os
import trimesh
import subprocess
from pathlib import Path
import meshio

import gyroid_utils
from gyroid_utils.utils import reload_all
reload_all()

working_path = os.getcwd()
print("Current working directory:", working_path)
parent_dir = os.path.dirname(working_path)



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[gyroid_utils] version 0.3.10 loaded
[gyroid_utils] version 0.3.10 loaded
gyroid_utils: all modules reloaded
Current working directory: g:\Limit\COFO\01 PhD research\202601 - generate gyroid\WP2 - stp files


In [ ]:
#==========================================================================================
#================================ load your STL file ======================================
#==========================================================================================
file_name = "gyroid-constant-1"
file_name = "gyroid-diagnal-1"
verts, faces = gyroid_utils.io_ops.load_stl(parent_dir + '/WP1 - stl files/' + file_name + '.stl')

# ------ visualize the quality of your mesh ----
triangle_areas = gyroid_utils.mesh_tools.calculate_triangle_areas(verts, faces)
gyroid_utils.viz.view_mesh(faces, verts)
gyroid_utils.viz.plot_histogram(triangle_areas)
gyroid_utils.mesh_tools.check_mesh_validity(verts,faces)


In [12]:
#==========================================================================================
#================================ convert STL to msh ======================================
#==========================================================================================

# define the path to ftetwild
ftetwild_exe = Path(
    r"C:\Program Files\fTetWild\build\Release\FloatTetwild_bin.exe"
)

# define the path to files
input_stl = Path(parent_dir + '/WP1 - stl files/' + file_name + '.stl')
output_msh = Path(working_path + '/' + file_name + '.msh')

# run ftetwild
cmd = [
    str(ftetwild_exe),
    "--input", str(input_stl),
    "--output", str(output_msh),
]

subprocess.run(cmd, check=True)
print("Meshing finished:", output_msh)


Meshing finished: g:\Limit\COFO\01 PhD research\202601 - generate gyroid\WP2 - stp files\gyroid-constant-1.msh


In [13]:
#==========================================================================================
#================================ convert msh to inp ======================================
#==========================================================================================
output_inp = Path(working_path + '/' + file_name + '.inp')

mesh = meshio.read(output_msh)
meshio.write(output_inp, mesh, file_format="abaqus")

print(f"Written {output_inp}")



Written g:\Limit\COFO\01 PhD research\202601 - generate gyroid\WP2 - stp files\gyroid-constant-1.inp
